# Lab 03 - Build Your Own Classifier

Build a K-Nearest Neighbors classifier from scratch with NumPy. Exercises marked with (*) are optional.

## Exercise 1 - Load Iris
Load the Iris dataset with `pandas.read_csv`. Use the UCI Iris file provided in the lab and inspect the data structure.

In [2]:
import tensorflow_datasets as tfds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [3]:
ds, info = tfds.load("iris", split="train", with_info=True)

raw_df = tfds.as_dataframe(ds, info)
df = pd.DataFrame(
    raw_df["features"].tolist(),
    columns=["sepal_length", "sepal_width", "petal_length", "petal_width"],
)
df["species"] = raw_df["label"].map(dict(enumerate(info.features["label"].names)))
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float32
 1   sepal_width   150 non-null    float32
 2   petal_length  150 non-null    float32
 3   petal_width   150 non-null    float32
 4   species       150 non-null    object 
dtypes: float32(4), object(1)
memory usage: 3.6+ KB


## Exercise 2 - Build the Test Split
Randomly select 20% of the records. Store the first four columns in `X_test` and the last column in `y_test`.

Store the remaining 80% of the records as `X_train` and `y_train`.

In [4]:
train_indexes = np.random.choice(len(df), int(len(df) * 0.8), replace=False)
x_train = df.iloc[train_indexes, :-1]
y_train = df.iloc[train_indexes, -1]
test_indexes = np.array(list(set(range(len(df))) - set(train_indexes)))
x_test = df.iloc[test_indexes, :-1]
y_test = df.iloc[test_indexes, -1]
x_train.shape, y_train.shape, x_test.shape, y_test.shape, df.shape

((120, 4), (120,), (30, 4), (30,), (150, 5))

## Exercise 4 - Implement KNN
Create a `KNearestNeighbors` class with `fit` and `predict`. Store only the data needed by the algorithm and decide whether all training samples must be kept.
Write NumPy implementations of the Euclidean, cosine, and Manhattan distances. Use Euclidean distance by default in the class.
Implement `predict` using the chosen distance metric and a majority vote over the `k` nearest neighbors.
Fit the model on `X_train` and `y_train`, predict `X_test`, and compute the accuracy against `y_test`.
Add weighted voting.

In [36]:
class KNearestNeighbors:
    def __init__(self, k):
        self.k = k
    def fit(self, x_train, y_train):
        self.x_train = x_train
        self.y_train = y_train
    def distance_algorithm(self, x, algorithm="euclidean"):
        if algorithm == "euclidean":
            return np.sqrt(np.sum((self.x_train - x) ** 2, axis = 1))
        elif algorithm == "manhattan":
            return np.sum(np.abs(self.x_train - x), axis = 1)
        elif algorithm == "cosine":
            return 1 - (self.x_train * x).sum(axis=1) / (np.linalg.norm(self.x_train, axis=1) * np.linalg.norm(x))
        else:
            raise ValueError("Invalid distance algorithm")
    def choose_label(self, nearest_labels, distances, weighted):
        if not weighted:
            return nearest_labels.mode()[0]
        else:
            weights = 1 / (distances + 1e-5)
            return nearest_labels.groupby(nearest_labels).apply(lambda x: weights[nearest_labels.index.isin(x.index)].sum()).idxmax()
    def predict(self, x_test, algorithm="euclidean", weighted = False):
        pred_labels = []
        for x in x_test.values:
            distances = self.distance_algorithm(x, algorithm)
            sorted_args = np.argsort(distances)
            nearest_labels = self.y_train.iloc[sorted_args[:self.k]]
            label = self.choose_label(nearest_labels, distances.iloc[sorted_args[:self.k]], weighted)
            pred_labels.append(label)
        return pred_labels
nearest_neighbour = KNearestNeighbors(3)
nearest_neighbour.fit(x_train, y_train)
predictions = nearest_neighbour.predict(x_test,"euclidean", True)
accuracy = (y_test.values == np.array(predictions)).sum() / len(y_test)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.68


## Exercise 9 - MNIST Test (*)
Apply the same KNN implementation to a reduced MNIST dataset with 100 samples per digit and evaluate the test accuracy.
Compare different values of `k`, `distance_metric`, and `weights` to find the best configuration on Iris and MNIST.

In [8]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
mnist_data = pd.DataFrame(mnist.data)
mnist_target = pd.Series(mnist.target)

mnist_data.shape, mnist_target.shape
x_test, x_train, y_test, y_train = train_test_split(mnist_data, mnist_target, test_size=0.8, random_state=42)
knearest_neighbour = KNearestNeighbors(3)
knearest_neighbour.fit(x_train, y_train)
kneast_neighbour_predictions = knearest_neighbour.predict(x_test[:100], "eucoloidean", True)
accuracy = (y_test[:100] == np.array(kneast_neighbour_predictions)).sum() / len(y_test[:100])
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.980


## Exercise 11 - Ames Housing Data
Load the Ames Housing dataset, keep only numerical features, choose `House Style` as the target, fill missing values with column means, and split into train and test sets.
Train KNN on Ames Housing and compare its accuracy with a naive majority-class baseline.

In [ ]:
!wget https://dbdmg.polito.it/dbdmg_web/wp-content/uploads/2025/10/ames_housing_dataset.zip
!tar -xf ames_housing_dataset.zip
!rm ames_housing_dataset.zip

--2026-05-04 21:11:41--  https://dbdmg.polito.it/dbdmg_web/wp-content/uploads/2025/10/ames_housing_dataset.zip
Resolving dbdmg.polito.it (dbdmg.polito.it)... 130.192.163.163
Connecting to dbdmg.polito.it (dbdmg.polito.it)|130.192.163.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 189065 (185K) [application/zip]
Saving to: 'ames_housing_dataset.zip.2'

     0K .......... .......... .......... .......... .......... 27%  417K 0s
    50K .......... .......... .......... .......... .......... 54%  778K 0s
   100K .......... .......... .......... .......... .......... 81% 2.08M 0s
   150K .......... .......... .......... ....                 100% 2.36M=0.2s

2026-05-04 21:11:41 (832 KB/s) - 'ames_housing_dataset.zip.2' saved [189065/189065]

'rm' is not recognized as an internal or external command,
operable program or batch file.


In [27]:
housing_df = pd.read_csv("AmesHousing.csv")
housing_df.dtypes
new_housing_df = housing_df.copy()
new_housing_df.drop(columns = housing_df.select_dtypes(include=["object", "datetime64[ns]"]).columns, inplace=True)
new_housing_df.drop(columns = ["PID", "Order"], inplace=True)
new_housing_df["HouseStyle"] = housing_df["House Style"].astype("category").cat.codes
new_housing_df.head()

,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,...,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice,HouseStyle
0,20,141.0,31770,6,5,1960,1960,112.0,639.0,0.0,...,62,0,0,0,0,0,5,2010,215000,2
1,20,80.0,11622,5,6,1961,1961,0.0,468.0,144.0,...,0,0,0,120,0,0,6,2010,105000,2
2,20,81.0,14267,6,6,1958,1958,108.0,923.0,0.0,...,36,0,0,0,0,12500,6,2010,172000,2
3,20,93.0,11160,7,5,1968,1968,0.0,1065.0,0.0,...,0,0,0,0,0,0,4,2010,244000,2
4,60,74.0,13830,5,5,1997,1998,0.0,791.0,0.0,...,34,0,0,0,0,0,3,2010,189900,5


In [46]:
x_train, x_test, y_train, y_test = train_test_split(new_housing_df.drop("HouseStyle", axis=1), new_housing_df["HouseStyle"], test_size=0.2, random_state=42)
knearest = KNearestNeighbors(5)
knearest.fit(x_train, y_train)
knearest_predictions = knearest.predict(x_test, "manhattan", True)
accuracy = (y_test.values == np.array(knearest_predictions)).sum() / len(y_test)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.78


## Exercise 13 - Standardization
Standardize the Ames features using the training-set mean and standard deviation, retrain KNN, and compare the accuracy.
Repeat the Ames experiment for different values of `k` and study how `k` affects performance.

In [57]:
standardized_df = (new_housing_df - new_housing_df.mean()) / new_housing_df.std()
x_train, x_test, y_train, y_test = train_test_split(standardized_df.drop("HouseStyle", axis=1), standardized_df["HouseStyle"], test_size=0.2, random_state=42)
knearest = KNearestNeighbors(13)
knearest.fit(x_train, y_train)
knearest_predictions = knearest.predict(x_test, "manhattan", True)
accuracy = (y_test.values == np.array(knearest_predictions)).sum() / len(y_test)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.91


## Exercise 15 - Stability Check (*)
Repeat the random train/test split multiple times, then compute the mean and standard deviation of the accuracy across runs.

In [59]:
from sklearn.model_selection import KFold
k_folds = 5
fold_size = len(new_housing_df) // k_folds
accuracies = []
for train_index, test_index in KFold(n_splits=k_folds, shuffle=True, random_state=42).split(new_housing_df):
    train_df = new_housing_df.iloc[train_index]
    test_df = new_housing_df.iloc[test_index]
    # Train and evaluate your model here
    # Append the accuracy to the accuracies list
    KNearest = KNearestNeighbors(5)
    KNearest.fit(train_df.drop("HouseStyle", axis=1), train_df["HouseStyle"])
    predictions = KNearest.predict(test_df.drop("HouseStyle", axis=1), "manhattan", True)
    accuracy = (test_df["HouseStyle"].values == np.array(predictions)).sum() / len(test_df)
    accuracies.append(accuracy)
print(f"Average Accuracy: {np.mean(accuracies):.2f}")



Average Accuracy: 0.76


## Exercise 16 - Distance Metrics
Rerun the experiments with Euclidean, cosine, and Manhattan distances, varying `k` and comparing the achieved accuracy.

In [61]:
from itertools import combinations
k = [3, 5, 7, 9, 11, 13]
algorithms = ["euclidean", "manhattan", "cosine"]
weighted = [True, False]
combinations = [(k_val, alg, w) for k_val in k for alg in algorithms for w in weighted]
results = []
for k_val, alg, w in combinations:
    KNearest = KNearestNeighbors(k_val)
    KNearest.fit(x_train, y_train)
    predictions = KNearest.predict(x_test, alg, w)
    accuracy = (y_test.values == np.array(predictions)).sum() / len(y_test)
    results.append((k_val, alg, w, accuracy))
results_df = pd.DataFrame(results, columns=["k", "algorithm", "weighted", "accuracy"])
print(results_df.sort_values(by="accuracy", ascending=False).head())

f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)
f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)
f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)
f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of N

     k  algorithm  weighted  accuracy
20   9  manhattan      True  0.907850
32  13  manhattan      True  0.906143
21   9  manhattan     False  0.904437
26  11  manhattan      True  0.901024
14   7  manhattan      True  0.899317


f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)
f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)
f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)
f:\Practices\MyRandomMLPractices\.venv\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of N